# Part 5 — Documents & Chunking

*The cut itself: how a raw document becomes the chunks we embed — and why that one decision quietly sets the ceiling for everything downstream.*

📖 Read the essay: https://www.mefby.com/essays/documents-and-chunking

🐍 Companion script: `chunking.py`

---

For four parts we kept saying *"the chunks"* as if our documents arrived pre-cut into tidy, self-contained pieces. They do not. A real source is a forty-page PDF, a sprawling web page, a thread of support messages. Someone — namely us — has to decide where to slice it. That decision is **chunking**, and it is one of the highest-leverage and most underrated steps in the whole pipeline: it sits at the very bottom of the stack, so its mistakes propagate up through everything. Garbage chunks in, garbage retrieval out.

**What you'll build, by hand:**

1. A small, eyeball-able sample document (the same store-policy world as the rest of the series).
2. **Fixed-size chunking** — blind cuts every `N` characters.
3. **Recursive character chunking** — try paragraph → line → sentence → word boundaries under a size cap (the sensible default).
4. **Sliding-window overlap** — the dial that keeps an idea on a seam from being sliced in two.
5. **Structure-aware chunking** — split on the document's *own* structure (Markdown headers).
6. **Semantic-ish chunking** — cut where neighboring sentences stop talking about the same thing.
7. **Metadata enrichment** — give each chunk its origin and a little context so it stands on its own.
8. A headline demo that runs all six lenses on one document so you can watch the boundaries move.

> **Runs fully offline.** This part is pure Python standard library — no models, no network, no API key. `numpy` is optional (used in exactly one tiny helper, and the demo runs without it). Where production would reach for a real library (LangChain's recursive splitter, a real embedding model for the semantic cut), that code is shown as a labelled **reference**, and a transparent pure-Python stand-in keeps every cell executable.

Previous: **Part 4 — Vector Databases & Indexing**. Next: **Part 6 — Build Your First RAG**.

## Setup

One import does the heavy lifting: `re`, for the bit of regex we need to find headers and split sentences. `numpy` is *optional* — we try to import it, and if it is missing we simply note that and carry on. (The later parts use this same "transparent fallback that runs without the heavy dependency" pattern; here the dependency is so light the demo never actually needs it.)

In [ ]:
from __future__ import annotations

import re

# numpy is OPTIONAL in this part. We try to import it; if it is missing we
# fall back to plain Python so the demo always runs. (Same 'transparent
# fallback that runs without the heavy dep' pattern the later parts use.)
try:
    import numpy as np  # noqa: F401
    HAVE_NUMPY = True
except Exception:  # numpy is installed here, but be safe.
    HAVE_NUMPY = False

print('numpy available:', HAVE_NUMPY, '(optional; everything below runs either way)')

## Step 0 — A small, eyeball-able sample document

Everything is easier to see on a document small enough to read in one glance. Ours is a tiny store policy — the same refunds / shipping / warranty world as Parts 2–4 — written with real Markdown headers (so the structure-aware splitter has something to follow) and a few distinct topics (so the semantic splitter has a real subject shift to find).

In [ ]:
SAMPLE_DOC = '''# Refund Policy

## Returns

Refunds are accepted within 30 days of purchase, provided the item is unused \
and in its original packaging. To start a return, email support@example.com \
with your order number. Refunds are processed within five business days of us \
receiving the item.

## Shipping

Standard shipping takes 3 to 5 business days. Express shipping arrives the next \
business day. Shipping fees are non-refundable. Items marked final sale cannot \
be returned or exchanged.

## Warranty

All electronics include a one-year limited warranty covering manufacturing \
defects. The warranty does not cover accidental damage or normal wear and tear.
'''

# The document's title, used later as a context prefix (the 'contextual' seed).
SAMPLE_TITLE = 'Refund Policy'

print(f'Sample document: {len(SAMPLE_DOC)} characters, title = {SAMPLE_TITLE!r}')
print()
print(SAMPLE_DOC)

We will print a lot of chunk lists, so here is one tiny helper to show each chunk with its length and a short preview. Seeing the lengths side by side is the fastest way to feel how a strategy behaves.

In [ ]:
def show(label, chunks, limit=70):
    """Print a list of chunks with their lengths and a short preview."""
    print(f'{label}  ->  {len(chunks)} chunk(s)')
    for i, c in enumerate(chunks):
        preview = c.replace('\n', '\\n')
        if len(preview) > limit:
            preview = preview[:limit] + '...'
        print(f'  [{i}] len={len(c):>3}  {preview!r}')
    print()

## Strategy 1 — Fixed-size chunking

The simplest, fastest, most predictable approach: split the text every `N` characters, full stop. It is also **blind** — it counts characters and nothing else, so it will happily cut through the middle of a sentence, even the middle of a word, because it has no idea those things exist. Good for a quick baseline; rough on meaning.

This is exactly the essay's worked one-liner, `[text[i:i+200] for i in range(0, len(text), 200)]`, generalized with a `stride`: how far we advance each step. `stride == size` gives adjacent, non-overlapping chunks; `stride < size` gives a sliding window (we will use that for overlap shortly).

In [ ]:
def fixed_size_chunks(text, size=200, stride=None):
    """Blind cuts every `size` chars. stride < size => a sliding window (overlap)."""
    if stride is None:
        stride = size  # advance by a full chunk => adjacent, non-overlapping
    return [text[i:i + size] for i in range(0, len(text), stride)]

Run it on the sample with a 200-character budget. Watch where the cuts land: a chunk ends mid-word (`pur` / `chase`), with zero respect for sentence or paragraph boundaries. That is the price of being fast and blind.

In [ ]:
fixed = fixed_size_chunks(SAMPLE_DOC, size=200)
show('fixed-size (size=200)', fixed)

## Strategy 2 — Recursive character chunking

Recursive chunking fixes most of that blindness. It tries a **hierarchy of separators**, coarsest first:

```python
["\n\n", "\n", ". ", " ", ""]   # paragraphs, lines, sentences, words, raw chars
```

It splits on the *coarsest* boundary that keeps pieces under the size cap, and only resorts to finer cuts when a piece is still too big. So it respects natural boundaries when it can. This is the sensible default for most prose — the thing most people should reach for first.

In production you would reach for the battle-tested library version (shown here as **reference** — we do not run it, to stay dependency-free):

```python
# THE INTENDED PATH (production)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=20,
    separators=["\n\n", "\n", ". ", " ", ""],
)
recursive = splitter.split_text(text)
```

We re-implement the core idea in pure Python so you can see every line of what that library is doing.

First, a small splitting helper. When we split on a separator we **keep it attached to the left piece**, so that re-joining pieces never silently drops the paragraph break / period / space we split on — chunk lengths stay honest. The empty separator `""` means "split into individual characters".

In [ ]:
DEFAULT_SEPARATORS = ['\n\n', '\n', '. ', ' ', '']

def _split_keep_separator(text, sep):
    """Split on `sep` but keep `sep` attached to the LEFT piece."""
    if sep == '':
        return list(text)  # '' => split into individual characters
    parts = text.split(sep)
    out = []
    for i, p in enumerate(parts):
        if i < len(parts) - 1:
            out.append(p + sep)  # reattach the separator we just split on
        elif p:                  # last piece: no trailing separator
            out.append(p)
    return out

Now the recursive splitter itself. For the current separator: cut the text into pieces, then greedily merge adjacent pieces back together while they still fit under `chunk_size`. Any single piece that is *still* too big can't be fixed by merging, so we hand it down to the next, finer separator and split again. That hand-down is the recursion — and the heart of why these chunks read like coherent thoughts.

In [ ]:
def recursive_split(text, chunk_size=200, separators=None):
    """Recursively split so every chunk is <= chunk_size where possible."""
    if separators is None:
        separators = DEFAULT_SEPARATORS

    # If it already fits, we are done. No need to cut a coherent thought.
    if len(text) <= chunk_size:
        return [text] if text else []

    sep = separators[0]
    finer = separators[1:]
    pieces = _split_keep_separator(text, sep)

    chunks = []
    current = ''
    for piece in pieces:
        # A piece bigger than the budget can't be fixed by merging: flush what
        # we have and recurse on it with a FINER separator.
        if len(piece) > chunk_size:
            if current:
                chunks.append(current)
                current = ''
            if finer:
                chunks.extend(recursive_split(piece, chunk_size, finer))
            else:
                # No finer separator left: hard-cut as a last resort.
                chunks.extend(piece[i:i + chunk_size]
                              for i in range(0, len(piece), chunk_size))
            continue

        # Otherwise, try to grow the current chunk with this piece.
        if len(current) + len(piece) <= chunk_size:
            current += piece
        else:
            if current:
                chunks.append(current)
            current = piece

    if current:
        chunks.append(current)
    return chunks

Same document, **same 200-character budget** as fixed-size — but look how different the result is. These chunks break at paragraph and sentence boundaries, so each one reads like a complete thought instead of a fragment cut off mid-word. Same size limit, very different chunks: that is the whole point.

In [ ]:
recursive = recursive_split(SAMPLE_DOC, chunk_size=200)
show('recursive (chunk_size=200)', recursive)

## Strategy 3 — Sliding-window overlap (the second dial)

Pick any strategy and you still have two **dials** to set. The first is **chunk size** — roughly how big each chunk is — which we have been setting all along. The second is **chunk overlap**, and it is the fix for chunk size's worst failure: when you cut a document into adjacent pieces, an idea that happens to straddle a boundary gets sliced in half, and *neither* chunk holds it whole.

Overlap solves this with a **sliding window**: consecutive chunks deliberately share some text at their seams, so a sentence sitting on a boundary survives intact in at least one chunk. A little overlap is cheap insurance against cutting an answer in two. The essay's rule of thumb: overlap of roughly 10–20% of the chunk size.

We add overlap *on top of* any pre-made chunk list, by prepending the tail of the previous chunk to the next one.

In [ ]:
def add_overlap(chunks, overlap=20):
    """Prepend the last `overlap` chars of each chunk to the next one."""
    if overlap <= 0 or len(chunks) < 2:
        return list(chunks)  # overlap == 0 => no-op (adjacent chunks)
    out = [chunks[0]]
    for prev, cur in zip(chunks, chunks[1:]):
        tail = prev[-overlap:]      # the seam we carry forward
        out.append(tail + cur)
    return out

Apply a 20-character overlap (about 10% of our 200-char budget) to the recursive chunks. Notice each chunk after the first now begins with a few characters borrowed from the end of its predecessor — the shared seam.

In [ ]:
overlapped = add_overlap(recursive, overlap=20)
show('recursive + overlap=20', overlapped)

To make the seam concrete: here is the exact 20-character tail carried forward from chunk `[0]` into chunk `[1]`. Any idea that landed on that boundary now lives whole inside chunk `[1]` — instead of being split between two and lost to both.

In [ ]:
if len(recursive) >= 2:
    seam = recursive[0][-20:]
    print('The 20-char seam carried forward from chunk [0] -> [1]:')
    print(' ', repr(seam))
    print('So an idea on the boundary survives whole in chunk [1].')

## Strategy 4 — Document-structure-aware chunking

Recursive splitting respects *generic* punctuation. Structure-aware splitting goes a step further and respects the document's **own** structure: Markdown headers, HTML sections, the functions in a source file, the slides in a deck. It honors the boundaries the author actually intended — which very often map perfectly onto coherent units.

Our document is Markdown, so we split on header lines (those starting with `#`). Each header begins a new section, and we keep the heading text so the chunk knows what it is about. We return dicts (`heading`, `level`, `text`) rather than bare strings, because the heading is structure worth keeping — it will feed both the metadata and the context prefix in the next step.

In [ ]:
HEADER_RE = re.compile(r'^(#{1,6})\s+(.*)$')

def structure_aware_split(text):
    """Split on Markdown headers, returning {heading, level, text} sections."""
    sections = []
    current = {'heading': None, 'level': 0, 'lines': []}

    def flush():
        body = '\n'.join(current['lines']).strip()
        if current['heading'] is not None or body:
            sections.append({
                'heading': current['heading'],
                'level': current['level'],
                'text': body,
            })

    for line in text.splitlines():
        m = HEADER_RE.match(line)
        if m:
            flush()                          # close the previous section
            level = len(m.group(1))          # number of '#' => heading depth
            current = {'heading': m.group(2).strip(), 'level': level, 'lines': []}
        else:
            current['lines'].append(line)
    flush()                                  # close the final section
    return sections

Split the sample. Each section lands on a heading the author wrote — `Returns`, `Shipping`, `Warranty` — so the cuts fall exactly where the meaning changes, for free, because the author already did the work of marking them.

In [ ]:
sections = structure_aware_split(SAMPLE_DOC)
for s in sections:
    head = s['heading'] if s['heading'] is not None else '(preamble)'
    body = s['text'].replace('\n', ' ')
    if len(body) > 60:
        body = body[:60] + '...'
    print(f"  h{s['level']} {head!r:<24} -> {body!r}")

## Strategy 5 — Semantic-ish chunking

The smartest strategy, and the slowest. Instead of looking at punctuation or structure, it looks at **meaning**: walk through the text keeping adjacent sentences together while they stay similar in topic, and cut precisely where that similarity drops. The boundaries land on genuine changes of subject.

The real version embeds each sentence with a model (Part 2) and measures cosine similarity between neighbors (Part 3). Shown here as **reference** — we do not run it, because that needs a model download and a GPU's worth of patience:

```python
# THE INTENDED PATH (production): cut where the EMBEDDING similarity drops
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
vecs = model.encode(sentences, normalize_embeddings=True)  # unit length
sims = [float(vecs[i] @ vecs[i + 1]) for i in range(len(vecs) - 1)]
# ...then cut where sims[i] falls below a threshold.
```

**Offline stand-in.** We have no model, so we approximate *"do these neighbors talk about the same thing?"* with a cheap, deterministic **lexical overlap** — the Jaccard similarity of their word sets. It is the same *shape* of signal: a number in `[0, 1]` that is high for on-topic neighbors and low at a topic shift. So the cutting logic is **identical** to the embedding version; only the scorer differs. (Be honest about the limit: a hash of shared words cannot tell that *refund* and *reimbursement* are cousins the way a real embedding can. It shows the mechanism, not the magic.)

First, a naive sentence splitter (split on `.`, `!`, `?` followed by whitespace) and the lexical similarity scorer that stands in for cosine-of-embeddings.

In [ ]:
def split_sentences(text):
    """Naive sentence split on ., !, ? followed by whitespace."""
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]


_WORD_RE = re.compile(r'[a-z0-9]+')

def _words(s):
    return set(_WORD_RE.findall(s.lower()))

def lexical_similarity(a, b):
    """Jaccard overlap of word sets: |A & B| / |A | B|, in [0, 1].

    Pure-Python stand-in for cosine similarity between two sentence embeddings.
    High when two sentences share vocabulary (likely same topic), low when
    they do not (likely a topic shift). Deterministic and offline.
    """
    wa, wb = _words(a), _words(b)
    if not wa and not wb:
        return 1.0
    inter = len(wa & wb)
    union = len(wa | wb)
    return inter / union if union else 0.0

Now the splitter: group adjacent sentences, and cut whenever neighbor similarity falls below a threshold. Chunk sizes will vary — that unpredictability is inherent to semantic chunking, and the price you pay for boundaries that land on real meaning rather than on a character count.

In [ ]:
def semantic_split(text, threshold=0.08):
    """Group adjacent sentences; cut where neighbor similarity < threshold."""
    sentences = split_sentences(text)
    if not sentences:
        return []

    chunks = []
    current = [sentences[0]]
    for prev, sent in zip(sentences, sentences[1:]):
        sim = lexical_similarity(prev, sent)
        if sim < threshold:
            # Similarity dropped: the subject changed here. Cut.
            chunks.append(' '.join(current))
            current = [sent]
        else:
            current.append(sent)
    chunks.append(' '.join(current))
    return chunks

Run it on the Returns + Shipping prose (joined together) so there is a real topic shift to find. First, the per-neighbor similarities, with the cut marked: you can literally read where the subject changes from refunds to shipping and watch the score dip below the threshold.

In [ ]:
prose = ' '.join(s['text'] for s in sections if s['text'])
sents = split_sentences(prose)
print('Adjacent-sentence similarities (low value = topic shift = a cut):')
for prev, nxt in zip(sents, sents[1:]):
    sim = lexical_similarity(prev, nxt)
    marker = '  <-- CUT' if sim < 0.08 else ''
    print(f'  sim={sim:.3f}  {prev[:34]!r:<38} | {nxt[:30]!r}{marker}')

In [ ]:
semantic = semantic_split(prose, threshold=0.08)
show('semantic-ish (threshold=0.08)', semantic)

## Step 6 — Metadata enrichment, and a little context

A chunk is more than its text. At split time, attach **metadata** — the source document, the section or heading, the page, the date, the author. This is not bookkeeping for its own sake: it is exactly what powers the **metadata filtering** from Part 4 ("search only documents tagged `2024`"), and it is what lets you show clean **citations** later ("this came from `refund-policy.md`, the Returns section").

There is a lighter trick too. An isolated chunk can lose the thread of what it is about, so prepending a little context — the document title and the section heading — keeps it self-explanatory. A chunk that begins `"Refund Policy > Returns: ..."` is far easier to retrieve correctly than the bare sentence alone. This is the seed of the *contextual* techniques later parts explore.

In [ ]:
def enrich(section, source, title):
    """Turn a {heading, text} section into a stored chunk with metadata and a
    context-prefixed `text` field ready to embed."""
    heading = section.get('heading')
    body = section.get('text', '')
    # Context prefix: title (> heading) so the chunk carries its own origin.
    prefix = title if not heading else f'{title} > {heading}'
    return {
        'text': f'{prefix}: {body}' if body else prefix,
        'raw_text': body,
        'metadata': {
            'source': source,
            'section': heading,
            'title': title,
        },
    }

Enrich the structure-aware sections. Each chunk now carries its origin (ready for citations and Part 4's filtering) *and* a heading prefix so it stays self-explanatory once it is retrieved, far from the document it came from.

In [ ]:
enriched = [enrich(s, source='refund-policy.md', title=SAMPLE_TITLE)
            for s in sections if s['text']]
for c in enriched:
    meta = c['metadata']
    print(f"  metadata: source={meta['source']!r} section={meta['section']!r}")
    prefixed = c['text'].replace('\n', ' ')
    if len(prefixed) > 64:
        prefixed = prefixed[:64] + '...'
    print(f'  text    : {prefixed!r}')
    print()

## Step 7 — The headline demo: six lenses, one document

This is the demo from `chunking.py`, run end to end. The same short document, cut six ways, so the trade-offs are impossible to miss. The cut is the foundation everything else (embeddings, similarity, retrieval) stands on — so watch how much the boundaries move depending on how you choose to make them.

In [ ]:
print('=' * 72)
print('Documents and Chunking, by hand (Part 5)')
print('numpy available:', HAVE_NUMPY, '(optional; demo runs either way)')
print('=' * 72)
print()

text = SAMPLE_DOC
print(f'Sample document: {len(text)} characters, title = {SAMPLE_TITLE!r}')
print()

# -- Strategy 1: fixed-size, blind cuts every 200 chars --------------------
print('-' * 72)
print('1) FIXED-SIZE (size=200): blind, cuts mid-sentence / mid-word')
print('-' * 72)
show('fixed-size', fixed_size_chunks(text, size=200))

# -- Strategy 2: recursive, prefers natural boundaries --------------------
print('-' * 72)
print('2) RECURSIVE (chunk_size=200): paragraph -> line -> sentence -> word')
print('-' * 72)
rec = recursive_split(text, chunk_size=200)
show('recursive', rec)
print('Note: same 200-char budget as fixed-size, but these break at natural')
print('boundaries, so each chunk reads like a coherent thought.')
print()

# -- Strategy 3: sliding-window overlap -----------------------------------
print('-' * 72)
print('3) OVERLAP (sliding window, overlap=20 ~= 10% of 200)')
print('-' * 72)
show('recursive + overlap', add_overlap(rec, overlap=20))

# -- Strategy 4: structure-aware (Markdown headers) -----------------------
print('-' * 72)
print('4) STRUCTURE-AWARE: split on Markdown headers (the author\'s own units)')
print('-' * 72)
for s in structure_aware_split(text):
    head = s['heading'] if s['heading'] is not None else '(preamble)'
    body = s['text'].replace('\n', ' ')
    if len(body) > 60:
        body = body[:60] + '...'
    print(f"  h{s['level']} {head!r:<24} -> {body!r}")
print()

# -- Strategy 5: semantic-ish (cut where similarity drops) ----------------
print('-' * 72)
print('5) SEMANTIC-ISH: cut where adjacent-sentence similarity drops')
print('   (offline stand-in: lexical Jaccard overlap in place of embeddings)')
print('-' * 72)
show('semantic-ish', semantic_split(prose, threshold=0.08))

# -- Step 6: metadata enrichment + context prefix -------------------------
print('-' * 72)
print('6) METADATA ENRICHMENT: attach source/section + a context prefix')
print('-' * 72)
for c in enriched:
    meta = c['metadata']
    print(f"  metadata: source={meta['source']!r} section={meta['section']!r}")
    prefixed = c['text'].replace('\n', ' ')
    if len(prefixed) > 64:
        prefixed = prefixed[:64] + '...'
    print(f'  text    : {prefixed!r}')
    print()

print('=' * 72)
print('Same document, six lenses. The cut is the foundation everything else')
print('(embeddings, similarity, retrieval) stands on.')
print('=' * 72)

## What you learned

- **Chunking** is one of the highest-leverage, most underrated steps in RAG. It sits at the bottom of the stack, so its mistakes propagate everywhere: garbage chunks in, garbage retrieval out, and no reranker or bigger model rescues a chunk that never held the answer.
- It starts with **document loading** — extracting clean text from messy formats (PDF columns and tables, HTML boilerplate, OCR on scans). Extraction quality is a hard ceiling on everything downstream.
- We chunk for three reasons: embedding **input limits**, retrieval **precision** (small chunks beat a blurry whole-document average), and generation-time **economy** (relevant chunks spend the context budget well).
- The central tension: too small loses context and turns ambiguous; too large blurs the embedding and wastes budget. Aim for **semantically coherent units**.
- The strategies climb a ladder: **fixed-size** (fast, blind) → **recursive character** (the sensible default) → **structure-aware** (follows the document's own boundaries) → **semantic** (cuts on meaning, slower). LLM-based chunking is the emerging frontier.
- The two dials are **chunk size** and **chunk overlap** (a sliding window that protects ideas on a boundary). The right values depend on your content and embedding model, so **measure, do not guess** — and attach **metadata** to every chunk.

**A note on what ran here:** every strategy above is pure standard library, so it executed with no model and no network. Where production would reach for a real library — LangChain's `RecursiveCharacterTextSplitter`, or a real embedding model for the semantic cut — we showed that code as a labelled reference and re-implemented the core idea by hand. The lexical-overlap scorer is an honest stand-in for cosine-of-embeddings: same shape of signal, none of the deep meaning. Swap in the real model and the cutting logic does not change at all.

## Next

We now hold every ingredient of a retrieval system: clean chunks (this part), embeddings (Part 2), similarity scoring (Part 3), and fast retrieval at scale (Part 4). In **Part 6 — Build Your First RAG**, we stop explaining and start building: assembling all of it into a complete, working RAG application, end to end.